In [ ]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from sklearn.metrics import accuracy_score, precision_score, recall_score

from msrest.authentication import CognitiveServicesCredentials
from array import array
import os
from PIL import Image, ImageDraw
import sys
import time
import matplotlib.pyplot as plt
from skimage import io
import numpy as np

BASE_IMAGE_PATH = r"D:\Facultate\Semestrul 4\Inteligenta Artificiala\Laboratoare\Lab 4\images\bikes"
OUTPUT_PATH = os.path.join(BASE_IMAGE_PATH, "output")

In [2]:
'''
Authenticate
Authenticates your credentials and creates a client.
'''
subscription_key = os.environ["VISION_KEY"]
endpoint = os.environ["VISION_ENDPOINT"]
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''

'\nEND - Authenticate\n'

In [3]:
def iou(boxA, boxB):
    xA_min, yA_min, xA_max, yA_max = boxA[0], boxA[1], boxA[0]+boxA[2], boxA[1]+boxA[3]
    xB_min, yB_min, xB_max, yB_max = boxB[0], boxB[1], boxB[0]+boxB[2], boxB[1]+boxB[3]

    xI_min = max(xA_min, xB_min)
    yI_min = max(yA_min, yB_min)
    xI_max = min(xA_max, xB_max)
    yI_max = min(yA_max, yB_max)

    inter_width = max(0, xI_max - xI_min)
    inter_height = max(0, yI_max - yI_min)
    inter_area = inter_width * inter_height

    boxA_area = boxA[2] * boxA[3]
    boxB_area = boxB[2] * boxB[3]

    union_area = boxA_area + boxB_area - inter_area
    if union_area == 0:
        return 0
    return inter_area / float(union_area)

In [5]:
def evaluate_multiple_detections(pred_boxes, gt_boxes, iou_threshold=0.5):
    matched_gt = [False] * len(gt_boxes)
    image_ious = []

    tp = 0
    fp = 0

    for pred_box in pred_boxes:
        best_iou, best_gt_idx = max(
            ((iou(pred_box, gt_box), idx) for idx, gt_box in enumerate(gt_boxes)),
            default=(0, -1),
            key=lambda x: x[0]
        )

        image_ious.append(best_iou)

        if best_iou >= iou_threshold:
            if not matched_gt[best_gt_idx]:
                tp += 1
                matched_gt[best_gt_idx] = True
            else:
                fp += 1 
        else:
            fp += 1

    fn = matched_gt.count(False)

    return tp, fp, fn, image_ious

In [6]:
manual_annotations = {
    'bike1.jpg': [[5, 30, 403, 376]],
    'bike02.jpg': [[17, 86, 362, 236]],
    'bike03.jpg': [[62, 144, 135, 250], [155, 140, 191, 257]],
    'bike04.jpg': [[0, 0, 415, 415]],
    'bike05.jpg': [[68, 39, 289, 306]],
    'bike06.jpg': [[62, 144, 135, 250], [155, 140, 191, 257]],
    'bike07.jpg': [[60, 185, 240, 230]],
    'bike08.jpg': [[42, 8, 346, 342]],
    'bike09.jpg': [[5, 7, 375, 373]],
    'bike10.jpg': [[140, 124, 237, 281]]
}

In [7]:
def process_images():
    print("Se proceseaza imaginile: ")

    y_true, y_pred = [], []
    total_tp = total_fp = total_fn = 0
    all_ious = []

    for root, _, files in os.walk(BASE_IMAGE_PATH):
        if "output" in root.lower():
            continue

        is_bike_folder = "bikes" in root.lower()

        for file in files:
            if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            img_path = os.path.join(root, file)
            print(f"\nProcesez imaginea: {file}")

            y_true.append(1 if is_bike_folder else 0)

            with open(img_path, "rb") as stream:
                analysis = computervision_client.analyze_image_in_stream(stream, visual_features=['Tags', 'Objects'])

            tags = {tag.name.lower() for tag in analysis.tags}
            has_bike = int(bool({'bicycle', 'bike'} & tags))
            y_pred.append(has_bike)

            detected_boxes = []
            img = Image.open(img_path)
            draw = ImageDraw.Draw(img)

            for obj in analysis.objects:
                if obj.object_property.lower() in {'bicycle', 'bike'}:
                    r = obj.rectangle
                    box = [r.x, r.y, r.w, r.h]
                    detected_boxes.append(box)

                    draw.rectangle(
                        [(r.x, r.y), (r.x + r.w, r.y + r.h)],
                        outline="red",
                        width=3
                    )
    
            output_file = os.path.join(OUTPUT_PATH, f"ai_detected_{file}")
            img.save(output_file)
            print(f"Imagine salvata: {output_file}")

            if file in manual_annotations:
                manual_boxes = manual_annotations[file]

                tp, fp, fn, img_ious = evaluate_multiple_detections(detected_boxes, manual_boxes)

                total_tp += tp
                total_fp += fp
                total_fn += fn
                all_ious.extend(img_ious)

                avg_iou = sum(img_ious) / len(img_ious) if img_ious else 0
                print(f"{tp} TP, {fp} FP, {fn} FN | IoU mediu: {avg_iou:.4f}")

    print("\nREZULTATE")

    print("\n1. Clasificare (bicicleta vs fara):")
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)

    print(f"  Accuracy:  {acc:.2f}")
    print(f"  Precision: {prec:.2f}")
    print(f"  Recall:    {rec:.2f}")

    print("\n2. Detectie (AI vs Manual):")

    if total_tp + total_fp + total_fn == 0:
        print("Nu exista date pentru detectie.")
        return

    detection_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
    detection_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
    mean_iou = sum(all_ious) / len(all_ious) if all_ious else 0

    print(f"TP: {total_tp} | FP: {total_fp} | FN: {total_fn}")
    print(f"  Precision: {detection_precision:.2f}")
    print(f"  Recall:    {detection_recall:.2f}")
    print(f"  Medie IoU: {mean_iou:.4f}")

In [8]:
process_images()

Se proceseaza imaginile: 

Procesez imaginea: bike02.jpg
Imagine salvata: D:\Facultate\Semestrul 4\Inteligenta Artificiala\Laboratoare\Lab 4\images\bikes\output\ai_detected_bike02.jpg
1 TP, 0 FP, 0 FN | IoU mediu: 0.9237

Procesez imaginea: bike03.jpg
Imagine salvata: D:\Facultate\Semestrul 4\Inteligenta Artificiala\Laboratoare\Lab 4\images\bikes\output\ai_detected_bike03.jpg
1 TP, 0 FP, 1 FN | IoU mediu: 0.8833

Procesez imaginea: bike04.jpg
Imagine salvata: D:\Facultate\Semestrul 4\Inteligenta Artificiala\Laboratoare\Lab 4\images\bikes\output\ai_detected_bike04.jpg
1 TP, 0 FP, 0 FN | IoU mediu: 0.9856

Procesez imaginea: bike05.jpg
Imagine salvata: D:\Facultate\Semestrul 4\Inteligenta Artificiala\Laboratoare\Lab 4\images\bikes\output\ai_detected_bike05.jpg
1 TP, 0 FP, 0 FN | IoU mediu: 0.9255

Procesez imaginea: bike06.jpg
Imagine salvata: D:\Facultate\Semestrul 4\Inteligenta Artificiala\Laboratoare\Lab 4\images\bikes\output\ai_detected_bike06.jpg
0 TP, 0 FP, 2 FN | IoU mediu: 0.0000